In [1]:
# Import packages
import os
import datetime
from datetime import datetime
import sys

sys.path.append("/Users/apple/repos/IKIEnv/lib/python3.7/site-packages/")

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 75)
import geopandas as gpd
#import shapely

from sqlalchemy import create_engine
from sqlalchemy import inspect, MetaData
import psycopg2
import multiprocess as mp
import gc

In [2]:
# import vessel data
vessels_2019 = pd.read_csv("/Users/apple/repos/datasets/vessels/vessels_2019.csv")

## Set-up Database Connection    

Turn voyage-based, geofenced inventory + fleet etc. code into a series of helper functions located in repos/datasets/inventories/ ???

In [3]:
host = 'nonprod-fuse-db-qcf-acw-analysis.coohyzo3ydfx.eu-west-1.rds.amazonaws.com'
user = 'fuse_master'
dbname = 'fuse'
userpass = open("/Users/apple/repos/datasets/etc/pw/part_a.txt","r").read() + open("/Users/apple/repos/datasets/etc/pw/part_b.txt","r").read()
port = 5432

# connect to cloud database
engine = create_engine('postgresql://{user}:{userpass}@{host}:{port}/{dbname}'.format(
    user=user,
    host=host,
    dbname=dbname,
    userpass=userpass,
    port=port,
    pool_size=10,
    max_overflow=2,
    pool_recycle=300,
    pool_pre_pring=True,
    pool_use_lifo=True))


# Import Voyage Data for Voyage-based Inventories

## Get Voyages Associated with Each Inventory

In [75]:
def get_inventory_voyages(country):
    
    # define columns to take down
    fc_per_voy_cols = ["source_iso", "source_name", "source_country", "dest_iso", "dest_name", "dest_country", "imo", "dist_travelled_ais", "energy_tj_voy", "energy_tj_stop", "co2_t"]
    voy_vess_cols = ["source_iso", "source_name", "source_country", "dest_iso", "dest_name", "dest_country", "imo", "type_bin", "ship_type_standardized", "capacity_unit", "size_bin", "gross_tonnage", "deadweight", "year_of_built", "dist_travelled_ais", "ene_tj", "co2_t"]
    
    # Domestic Voyages
    dom_voys_raw = pd.read_sql(r"SELECT {0} FROM fuse_2018.voyages_foc WHERE dest_country = '{1}' AND source_country = '{1}'".format(", ".join(fc_per_voy_cols), country), engine)
    dom_voys_raw["ene_tj"] = dom_voys_raw.energy_tj_voy + dom_voys_raw.energy_tj_stop
    dom_voys_vess = pd.merge(dom_voys_raw.drop(columns=["energy_tj_voy", "energy_tj_stop"]), vessels_2019, left_on="imo", right_on="imo", how="left")[voy_vess_cols]

    # Internationally Departing Voyages
    int_dep_voys_raw = pd.read_sql(r"SELECT {0} FROM fuse_2018.voyages_foc WHERE source_country = '{1}' AND dest_country != '{1}'".format(", ".join(fc_per_voy_cols), country), engine)
    int_dep_voys_raw["ene_tj"] = int_dep_voys_raw.energy_tj_voy + int_dep_voys_raw.energy_tj_stop
    int_dep_voys_vess = pd.merge(int_dep_voys_raw.drop(columns=["energy_tj_voy", "energy_tj_stop"]), vessels_2019, left_on="imo", right_on="imo", how="left")[voy_vess_cols]

    # Internationally Arriving Voyages
    int_arr_voys_raw = pd.read_sql(r"SELECT {0} FROM fuse_2018.voyages_foc WHERE dest_country = '{1}' AND source_country != '{1}'".format(", ".join(fc_per_voy_cols), country), engine)
    int_arr_voys_raw["ene_tj"] = int_arr_voys_raw.energy_tj_voy + int_arr_voys_raw.energy_tj_stop
    int_arr_voys_vess = pd.merge(int_arr_voys_raw.drop(columns=["energy_tj_voy", "energy_tj_stop"]), vessels_2019, left_on="imo", right_on="imo", how="left")[voy_vess_cols]
    
    return dom_voys_vess, int_dep_voys_vess, int_arr_voys_vess

# Prepare Voyage Sets for Each Inventory

What columns will we need here such that the following are all derivable, later to be disaggregated in terms of ports and vessel types?:    
<list>
<li> Number of unique vessels
<li> Median build year (fleet average, voyages average and transport supply average)
<li> Transport supply
<li> Median voyage distance (fleet average, voyages average and transport supply average)
<li> Energy (TJ)
<li> CO2 (t)
</list>

In [51]:
def prepare_inventory_voyage_sets(dom_voys_vess, int_dep_voys_vess, int_arr_voys_vess):
    
    inv_cols = ["source_iso", "source_name", "source_country", "dest_iso", "dest_name", "dest_country", "imo", "type_bin", "ship_type_standardized", "capacity_unit", "size_bin", "gross_tonnage", "deadweight", "year_of_built", "dist_travelled_ais", "ene_tj", "co2_t"]
    
    # Domestic
    dom_voys = dom_voys_vess.dropna(axis=0, how='any')
    dom_voys["gt_nm"] = dom_voys.gross_tonnage * dom_voys.dist_travelled_ais
    dom_voys["dwt_nm"] = dom_voys.deadweight * dom_voys.dist_travelled_ais
    dom_voys["gt_nm_X_bld_yr"] = dom_voys.gt_nm * dom_voys.year_of_built
    dom_voys["dwt_nm_X_bld_yr"] = dom_voys.dwt_nm * dom_voys.year_of_built

    # International Departures
    int_dep_voys = int_dep_voys_vess.dropna(axis=0, how='any')
    int_dep_voys["gt_nm"] = int_dep_voys.gross_tonnage * int_dep_voys.dist_travelled_ais
    int_dep_voys["dwt_nm"] = int_dep_voys.deadweight * int_dep_voys.dist_travelled_ais
    int_dep_voys["gt_nm_X_bld_yr"] = int_dep_voys.gt_nm * int_dep_voys.year_of_built
    int_dep_voys["dwt_nm_X_bld_yr"] = int_dep_voys.dwt_nm * int_dep_voys.year_of_built

    # International Arrivals
    int_arr_voys = int_arr_voys_vess.dropna(axis=0, how='any')
    int_arr_voys["gt_nm"] = int_arr_voys.gross_tonnage * int_arr_voys.dist_travelled_ais
    int_arr_voys["dwt_nm"] = int_arr_voys.deadweight * int_arr_voys.dist_travelled_ais
    int_arr_voys["gt_nm_X_bld_yr"] = int_arr_voys.gt_nm * int_arr_voys.year_of_built
    int_arr_voys["dwt_nm_X_bld_yr"] = int_arr_voys.dwt_nm * int_arr_voys.year_of_built

    return dom_voys, int_dep_voys, int_arr_voys
    

# Aggregate into Inventories

#### Define Aggregation Helper Function

In [52]:
def get_voy_stats(aggregator, voys_set):
    """
    Define function to take stats associated with each port.
    """
    
    ## Number of voyages recorded and number of vessels in executing fleet
    n_vys = len(voys_set)
    n_imo = len(voys_set.imo.unique())
    
    ## Average Build Year - mean chosen as build year is robust and not easily skewed
    aby_vys = voys_set.year_of_built.mean()
    aby_flt = voys_set.groupby(["imo"]).agg({"year_of_built":"median"}).year_of_built.mean()
    aby_gt_nm = voys_set.gt_nm_X_bld_yr.sum() / voys_set.gt_nm.sum() # weighted mean
    aby_dwt_nm = voys_set.dwt_nm_X_bld_yr.sum() / voys_set.dwt_nm.sum() # weighted mean
    
    ## Transport Supply
    tps_gt_nm = voys_set.gt_nm.sum()
    tps_dwt_nm = voys_set.dwt_nm.sum()
    
    ## Average Voyage Distance - median chosen to counteract mitigate inaccuracies in dist_travelled_ais metric (a lot observed in Summer 2023)
    avd_vys = voys_set.dist_travelled_ais.median()
    avd_flt = voys_set.groupby(["imo"]).agg({"dist_travelled_ais":"median"}).dist_travelled_ais.mean() # 'mean of median' values
    avd_gt_nm = voys_set.gt_nm.sum() / voys_set.gross_tonnage.sum() # weighted mean
    avd_dwt_nm = voys_set.dwt_nm.sum() / voys_set.deadweight.sum() # weighted mean
    
    ## Inventory of Energy Demand and CO2 Generation
    ene_tj = voys_set.ene_tj.sum()
    co2_t = voys_set.co2_t.sum()
    
    # combine into single results dataframe
    res_row = [aggregator, n_vys, n_imo, aby_vys, aby_flt, aby_gt_nm, aby_dwt_nm, tps_gt_nm, tps_dwt_nm, avd_vys, avd_flt, avd_gt_nm, avd_dwt_nm, ene_tj, co2_t]
    
    return res_row

## Domestic Activity

### Aggregation by Port

In [30]:
def dom_port_agg(dom_voys):

    res_cols = ["port_pair", "n_vys", "n_imo", "aby_vys", "aby_flt", "aby_gt_nm", "aby_dwt_nm", "tps_gt_nm", "tps_dwt_nm", "avd_vys", "avd_flt", "avd_gt_nm", "avd_dwt_nm", "ene_tj", "co2_t"]
    
    dom_port_res_df = pd.DataFrame(columns=res_cols)
    port_pairs_all = dom_voys[["source_name", "dest_name"]]
    port_pairs_set = port_pairs_all[~port_pairs_all.apply(frozenset, axis=1).duplicated()]
    
    for idx, pair in port_pairs_set.iterrows():

        # take all voyages associated with the port-pair
        outgoing = dom_voys[(dom_voys.source_name == pair.source_name) & (dom_voys.dest_name == pair.dest_name)]
        incoming = dom_voys[(dom_voys.source_name == pair.dest_name) & (dom_voys.dest_name == pair.source_name)]
        dom_voys_port_pair = pd.concat([outgoing, incoming])

        # combine into single results dataframe
        dom_port_res_df.loc[dom_port_res_df.shape[0]] = get_voy_stats(str(pair.source_name) + " - " + str(pair.dest_name), dom_voys_port_pair)

    dom_port_res_df_clean = dom_port_res_df.fillna(0).round(0).astype({"aby_vys":"int", "aby_flt":"int", "aby_gt_nm":"int", "aby_dwt_nm":"int", "tps_gt_nm":"int", "tps_dwt_nm":"int", "avd_vys":"int", "avd_flt":"int", "avd_gt_nm":"int", "avd_dwt_nm":"int", "ene_tj":"int", "co2_t":"int"})
    dom_port_res_df_clean.sort_values(by="ene_tj", ascending=False)

    return dom_port_res_df_clean

### Aggregation by Vessel Type

In [29]:
def dom_type_agg(dom_voys):
    
    res_cols = ["dom_type", "n_vys", "n_imo", "aby_vys", "aby_flt", "aby_gt_nm", "aby_dwt_nm", "tps_gt_nm", "tps_dwt_nm", "avd_vys", "avd_flt", "avd_gt_nm", "avd_dwt_nm", "ene_tj", "co2_t"]
    dom_type_res_df = pd.DataFrame(columns=res_cols)

    for vessel_type in dom_voys.ship_type_standardized.unique():

        # take voyages associated with each port
        dom_voys_type = dom_voys[(dom_voys.ship_type_standardized == vessel_type)]

        # combine into single results dataframe
        dom_type_res_df.loc[dom_type_res_df.shape[0]] = get_voy_stats(vessel_type, dom_voys_type)

    dom_type_res_df_clean = dom_type_res_df.fillna(0).round(0).astype({"aby_vys":"int", "aby_flt":"int", "aby_gt_nm":"int", "aby_dwt_nm":"int", "tps_gt_nm":"int", "tps_dwt_nm":"int", "avd_vys":"int", "avd_flt":"int", "avd_gt_nm":"int", "avd_dwt_nm":"int", "ene_tj":"int", "co2_t":"int"})
    dom_type_res_df_clean.sort_values(by="ene_tj", ascending=False)
    
    return dom_type_res_df_clean

## International Departures

### Aggregation by Port

In [28]:
def int_dep_port_agg(int_dep_voys):
    
    res_cols = ["int_dep_port", "n_vys", "n_imo", "aby_vys", "aby_flt", "aby_gt_nm", "aby_dwt_nm", "tps_gt_nm", "tps_dwt_nm", "avd_vys", "avd_flt", "avd_gt_nm", "avd_dwt_nm", "ene_tj", "co2_t"]
    int_deps_port_res_df = pd.DataFrame(columns=res_cols)

    for source_port in int_dep_voys.source_name.unique():

        # take voyages associated with each port
        int_dep_voys_port = int_dep_voys[(int_dep_voys.source_name == source_port)]

        # combine into single results dataframe
        int_deps_port_res_df.loc[int_deps_port_res_df.shape[0]] = get_voy_stats(source_port, int_dep_voys_port)

    int_deps_port_res_df_clean = int_deps_port_res_df.fillna(0).round(0).astype({"aby_vys":"int", "aby_flt":"int", "aby_gt_nm":"int", "aby_dwt_nm":"int", "tps_gt_nm":"int", "tps_dwt_nm":"int", "avd_vys":"int", "avd_flt":"int", "avd_gt_nm":"int", "avd_dwt_nm":"int", "ene_tj":"int", "co2_t":"int"})
    int_deps_port_res_df_clean.sort_values(by="ene_tj", ascending=False)

    return int_deps_port_res_df_clean

### Aggregation by Vessel Type

In [27]:
def int_dep_type_agg(int_dep_voys):
    
    res_cols = ["int_dep_type", "n_vys", "n_imo", "aby_vys", "aby_flt", "aby_gt_nm", "aby_dwt_nm", "tps_gt_nm", "tps_dwt_nm", "avd_vys", "avd_flt", "avd_gt_nm", "avd_dwt_nm", "ene_tj", "co2_t"]
    int_deps_type_res_df = pd.DataFrame(columns=res_cols)

    for vessel_type in int_dep_voys.ship_type_standardized.unique():

        # take voyages associated with each port
        int_dep_voys_type = int_dep_voys[(int_dep_voys.ship_type_standardized == vessel_type)]

        # combine into single results dataframe
        int_deps_type_res_df.loc[int_deps_type_res_df.shape[0]] = get_voy_stats(vessel_type, int_dep_voys_type)

    int_deps_type_res_df_clean = int_deps_type_res_df.fillna(0).round(0).astype({"aby_vys":"int", "aby_flt":"int", "aby_gt_nm":"int", "aby_dwt_nm":"int", "tps_gt_nm":"int", "tps_dwt_nm":"int", "avd_vys":"int", "avd_flt":"int", "avd_gt_nm":"int", "avd_dwt_nm":"int", "ene_tj":"int", "co2_t":"int"})
    int_deps_type_res_df_clean.sort_values(by="ene_tj", ascending=False)
    
    return int_deps_type_res_df_clean

## International Arrivals

### Aggregation by Port

In [26]:
def int_arr_port_agg(int_arr_voys):
    
    res_cols = ["int_arr_port", "n_vys", "n_imo", "aby_vys", "aby_flt", "aby_gt_nm", "aby_dwt_nm", "tps_gt_nm", "tps_dwt_nm", "avd_vys", "avd_flt", "avd_gt_nm", "avd_dwt_nm", "ene_tj", "co2_t"]
    int_arrs_port_res_df = pd.DataFrame(columns=res_cols)

    for dest_port in int_arr_voys.dest_name.unique():

        # take voyages associated with each port
        int_arr_voys_port = int_arr_voys[(int_arr_voys.dest_name == dest_port)]

        # combine into single results dataframe
        int_arrs_port_res_df.loc[int_arrs_port_res_df.shape[0]] = get_voy_stats(dest_port, int_arr_voys_port)

    int_arrs_port_res_df_clean = int_arrs_port_res_df.fillna(0).round(0).astype({"aby_vys":"int", "aby_flt":"int", "aby_gt_nm":"int", "aby_dwt_nm":"int", "tps_gt_nm":"int", "tps_dwt_nm":"int", "avd_vys":"int", "avd_flt":"int", "avd_gt_nm":"int", "avd_dwt_nm":"int", "ene_tj":"int", "co2_t":"int"})
    int_arrs_port_res_df_clean.sort_values(by="ene_tj", ascending=False)
    
    return int_arrs_port_res_df_clean

### Aggregation by Vessel Type

In [25]:
def int_arr_type_agg(int_arr_voys):

    res_cols = ["int_arr_type", "n_vys", "n_imo", "aby_vys", "aby_flt", "aby_gt_nm", "aby_dwt_nm", "tps_gt_nm", "tps_dwt_nm", "avd_vys", "avd_flt", "avd_gt_nm", "avd_dwt_nm", "ene_tj", "co2_t"]
    int_arrs_type_res_df = pd.DataFrame(columns=res_cols)

    for vessel_type in int_arr_voys.ship_type_standardized.unique():

        # take voyages associated with each port
        int_arr_voys_type = int_arr_voys[(int_arr_voys.ship_type_standardized == vessel_type)]

        # combine into single results dataframe
        int_arrs_type_res_df.loc[int_arrs_type_res_df.shape[0]] = get_voy_stats(vessel_type, int_arr_voys_type)

    int_arrs_type_res_df_clean = int_arrs_type_res_df.fillna(0).round(0).astype({"aby_vys":"int", "aby_flt":"int", "aby_gt_nm":"int", "aby_dwt_nm":"int", "tps_gt_nm":"int", "tps_dwt_nm":"int", "avd_vys":"int", "avd_flt":"int", "avd_gt_nm":"int", "avd_dwt_nm":"int", "ene_tj":"int", "co2_t":"int"})
    int_arrs_type_res_df_clean.sort_values(by="ene_tj", ascending=False)
    
    return int_arrs_type_res_df_clean

# Run

In [14]:
# set country name
country_list_raw = pd.read_sql(r"SELECT DISTINCT dest_country FROM fuse_2018.voyages_foc", engine).dest_country.unique()
country_list = pd.Series(country_list_raw).sort_values().values

country_list_a = country_list[0:40]
country_list_b = country_list[40:80]
country_list_c = country_list[80:120]
country_list_d = country_list[120:160]
country_list_e = country_list[160:200]
country_list_f = country_list[200:]

In [133]:
country_list_test = country_list_f

In [134]:
country_list_test

array(['Uruguay', 'Vanuatu', 'Venezuela', 'Viet Nam',
       'Wallis and Futuna Islands', 'Western Sahara'], dtype=object)

In [135]:
country_list_chunk = country_list_test

for country_idx in range(len(country_list_test)):
    
    country = country_list_chunk[country_idx]
    print("Processing {0} ({1} of {2}).".format(country, country_idx + 1, len(country_list_chunk)))
        
    # check if folder at directory exists, if not create it
    country_dir = "/Users/apple/repos/datasets/inventories/worldwide/{0}/".format(country)
    os.makedirs(country_dir, exist_ok=True)

    # Get voyages associated with each inventory
    dom_voys_vess, int_dep_voys_vess, int_arr_voys_vess = get_inventory_voyages(country)

    # Prepare/format voyage sets associated with each inventory
    dom_voys, int_dep_voys, int_arr_voys = prepare_inventory_voyage_sets(dom_voys_vess, int_dep_voys_vess, int_arr_voys_vess)

    # Aggregate into Inventories
    ## Domestic Aggregated to Port
    dom_port_res_df_clean = dom_port_agg(dom_voys)
    dom_port_res_df_clean.to_csv(country_dir + "dom_inv_by_port.csv", index=False)

    ## Domestic Aggregated to Vessel Type
    dom_type_res_df_clean = dom_type_agg(dom_voys)
    dom_type_res_df_clean.to_csv(country_dir + "dom_inv_by_vess_type.csv", index=False)

    ## International Departures Aggregated to Port
    int_deps_port_res_df_clean = int_dep_port_agg(int_dep_voys)
    int_deps_port_res_df_clean.to_csv(country_dir + "int_dep_inv_by_port.csv", index=False)

    ## International Departures Aggregated to Vessel Type
    int_deps_type_res_df_clean = int_dep_type_agg(int_dep_voys)
    int_deps_type_res_df_clean.to_csv(country_dir + "int_dep_inv_by_vess_type.csv", index=False)

    ## International Arrivals Aggregated to Port
    int_arrs_port_res_df_clean = int_arr_port_agg(int_arr_voys)
    int_arrs_port_res_df_clean.to_csv(country_dir + "int_arr_inv_by_port.csv", index=False)

    ## International Arrivals Aggregated to Vessel Type
    int_arrs_type_res_df_clean = int_arr_type_agg(int_arr_voys)
    int_arrs_type_res_df_clean.to_csv(country_dir + "int_arr_inv_by_vess_type.csv", index=False)


Processing Uruguay (1 of 6).
Processing Vanuatu (2 of 6).
Processing Venezuela (3 of 6).
Processing Viet Nam (4 of 6).


/Users/apple/repos/deepStock/env/lib/python3.7/site-packages/ipykernel_launcher.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/apple/repos/deepStock/env/lib/python3.7/site-packages/ipykernel_launcher.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/apple/repos/deepStock/env/lib/python3.7/site-packages/ipykernel_launcher.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

Processing Wallis and Futuna Islands (5 of 6).
Processing Western Sahara (6 of 6).
